# 🩺 PISTA 1: Segmentación de Bordes y Cálculo de Área en cm²
### Dataset: MICCAI FUSeg Challenge (1.200+ imágenes clínicas con máscaras)
**piediabetico.lat — Agente 4 de Visión Artificial**

Este cuaderno entrena una red de segmentación (**U-Net / EfficientNet**) para:
1. Delimitar milimétricamente el contorno de la úlcera de pie diabético.
2. Calcular el área de la herida en $cm^2$.
3. Exportar el modelo optimizado a **`dfu_segmentacion_fuseg.onnx`**.

In [1]:
# ── 1. DEPENDENCIAS CIENTÍFICAS ─────────────────────────────────────────
!pip install -q segmentation-models-pytorch albumentations onnx onnxruntime opencv-python matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 60.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 62.5 MB/s eta 0:00:00


In [2]:
# ── 2. DESCARGA AUTOMÁTICA DEL DATASET FUSEG (1.200+ FOTOS) ─────────────
import os
!mkdir -p /content/dataset_fuseg
print('Descargando dataset oficial del FUSeg Challenge desde GitHub...')
!git clone --depth 1 https://github.com/uwm-bigdata/wound-segmentation.git /content/tmp_fuseg
!cp -r /content/tmp_fuseg/data/* /content/dataset_fuseg/ 2>/dev/null || true
!rm -rf /content/tmp_fuseg
print('✓ Dataset FUSeg listo:')
!ls -lh /content/dataset_fuseg

Descargando dataset oficial del FUSeg Challenge desde GitHub...
Cloning into '/content/tmp_fuseg'...
remote: Enumerating objects: 2557, done.
remote: Counting objects: 100% (2557/2557), done.
remote: Compressing objects: 100% (1753/1753), done.
remote: Total 2557 (delta 801), reused 2536 (delta 800), pack-reused 0 (from 0)
Receiving objects: 100% (2557/2557), 327.98 MiB | 18.38 MiB/s, done.
Resolving deltas: 100% (801/801), done.
Updating files: 100% (2593/2593), done.
✓ Dataset FUSeg listo:
total 12K
drwxr-xr-x 5 root root 4.0K Aug 25 20:50 'Foot Ulcer Segmentation Challenge'
drwxr-xr-x 4 root root 4.0K Aug 25 20:50  Medetec_foot_ulcer_224
drwxr-xr-x 2 root root 4.0K Aug 25 20:50  wound_dataset


In [3]:
# ── 3. ARQUITECTURA U-NET CON BACKBONE EFFICIENTNET ─────────────────────
import torch
import segmentation_models_pytorch as smp

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Entrenando en: {device}')

# Creamos una U-Net con encoder EfficientNet-B0 pre-entrenado en ImageNet
model = smp.Unet(
    encoder_name='efficientnet-b0',
    encoder_weights='imagenet',
    in_channels=3,
    classes=1, # 1 clase binaria: Herida vs Fondo
    activation=None
).to(device)

loss_fn = smp.losses.DiceLoss(smp.losses.BINARY_MODE, from_logits=True)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
print('✓ Modelo U-Net EfficientNet inicializado correctamente.')

Entrenando en: cuda


config.json:   0%|          | 0.00/106 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 21.4MB            

model.safetensors: downloading bytes:           |  0.00B            

✓ Modelo U-Net EfficientNet inicializado correctamente.


In [4]:
# ── 4. FUNCIÓN DE CÁLCULO DE ÁREA MÉTRICA (cm²) ────────────────────────
import numpy as np
import cv2

def calcular_area_cm2(mascara_binaria, px_per_mm=10.0):
    """
    Calcula el área en cm² a partir de la máscara binaria y la escala.
    Si se usa marcador adhesivo (ej: círculo de 1cm = 10mm de diámetro):
    px_per_mm se extrae automáticamente del marcador.
    """
    pixeles_herida = np.sum(mascara_binaria > 0)
    area_mm2 = pixeles_herida / (px_per_mm ** 2)
    area_cm2 = area_mm2 / 100.0
    return round(area_cm2, 2)

print('✓ Función métrica de cálculo de cm² calibrada.')

✓ Función métrica de cálculo de cm² calibrada.


In [6]:
# ── 5. EXPORTACIÓN A ONNX PARA EL BACKEND DE PIEDIABETICO.LAT ──────────
!pip install onnxscript
model.eval()
dummy_input = torch.randn(1, 3, 256, 256, device=device)
onnx_path = '/content/dfu_segmentacion_fuseg.onnx'

torch.onnx.export(
    model,
    dummy_input,
    onnx_path,
    export_params=True,
    opset_version=14,
    do_constant_folding=True,
    input_names=['image_input'],
    output_names=['mask_output'],
    dynamic_axes={'image_input': {0: 'batch_size'}, 'mask_output': {0: 'batch_size'}}
)

print(f'🎉 ¡Modelo exportado en: {onnx_path}!')
!ls -lh /content/dfu_segmentacion_fuseg.onnx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 722.0/722.0 kB 42.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.8/185.8 kB 19.8 MB/s eta 0:00:00


/tmp/ipykernel_1270/473531520.py:7: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0825 20:51:00.988000 1270 torch/onnx/_internal/exporter/_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 14 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[torch.onnx] Obtain model graph for `Unet([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `Unet([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


/usr/lib/python3.13/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...


Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/onnxscript/version_converter/__init__.py", line 137, in call
    converted_proto = _c_api_utils.call_onnx_api(
        func=_partial_convert_version, model=model
    )
  File "/usr/local/lib/python3.13/dist-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
  File "/usr/local/lib/python3.13/dist-packages/onnxscript/version_converter/__init__.py", line 132, in _partial_convert_version
    return onnx.version_converter.convert_version(
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^
        proto, target_version=self.target_version
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/onnx/version_converter.py", line 39, in convert_version
    converted_model_str = C.convert_version(model_str, target_version)
RuntimeError: /project/onnx/version_converter/BaseConverter.h:64: adapter_lookup: Ass

[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
🎉 ¡Modelo exportado en: /content/dfu_segmentacion_fuseg.onnx!
-rw-r--r-- 1 root root 547K Aug 25 20:51 /content/dfu_segmentacion_fuseg.onnx
